# <font color="steelblue">Despliegue de modelos con Dash</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 10/06/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

---

## Cómo usar este cuaderno

Los módulos 0 a 2 son de puesta en marcha. Los módulos 3 a 7 son la referencia del lenguaje: layout, componentes, callbacks y los patrones que evitan que la aplicación se vuelva lenta. El módulo 8 construye un cuadro de mando completo paso a paso, y el 9 lo publica en internet.

Si vienes del curso de Streamlit, presta especial atención al **módulo 0**: el modelo de ejecución de Dash es el opuesto, y arrastrar los hábitos de Streamlit es la principal fuente de confusión.

---

# Índice

| Módulo | Contenido |
|---|---|
| **0** | Qué es Dash y cómo piensa |
| **1** | Instalación y primera aplicación |
| **2** | Ejecución desde Google Colab |
| **3** | El layout: construir la página con Python |
| **4** | Componentes: `dcc`, `html` y Bootstrap |
| **5** | Callbacks: el corazón de Dash |
| **6** | Gráficos con Plotly |
| **7** | Tablas, estado y rendimiento |
| **8** | Construcción guiada de un cuadro de mando |
| **9** | **Publicación en internet** |
| **10** | Buenas prácticas y catálogo de errores |
| **A** | Apéndice: chuleta de referencia |
| **B** | Apéndice: comparativa con Streamlit y Gradio |

---

# Módulo 0. Qué es Dash y cómo piensa

## 0.1 Qué es exactamente

Dash es un framework de Plotly para construir **aplicaciones web analíticas** escribiendo solo Python. Por debajo son tres piezas cosidas:

- **Flask** en el servidor, que atiende las peticiones HTTP.
- **React** en el navegador, que dibuja la interfaz.
- **Plotly.js** para los gráficos.

Lo relevante es que tú no tocas ninguna de las tres. Escribes Python, y Dash traduce.

## 0.2 La idea central

> **Una aplicación de Dash es un árbol de componentes (el *layout*) más un conjunto de funciones (los *callbacks*) que se disparan cuando cambian propiedades concretas de esos componentes.**

Dos piezas, y conviene tenerlas separadas mentalmente:

**El layout** describe *qué hay* en la página. Es una estructura de datos de Python que se traduce a HTML:

```python
app.layout = html.Div([
    dcc.Slider(id="mi-slider", min=0, max=10, value=5),
    html.Div(id="mi-salida"),
])
```

**Los callbacks** describen *qué pasa* cuando algo cambia:

```python
@callback(
    Output("mi-salida", "children"),
    Input("mi-slider", "value"),
)
def actualizar(valor):
    return f"Has elegido {valor}"
```

Se lee así: «cuando cambie la propiedad `value` del componente `mi-slider`, ejecuta esta función y pon lo que devuelva en la propiedad `children` del componente `mi-salida`».

## 0.3 La diferencia fundamental con Streamlit

Esto es lo más importante del módulo, y merece detenerse.

En **Streamlit**, el script entero se re-ejecuta de arriba abajo cada vez que el usuario toca algo. Es simple de entender, pero obliga a usar caché para cualquier cosa costosa, porque todo se recalcula constantemente.

En **Dash**, el script se ejecuta **una sola vez, al arrancar el servidor**. A partir de ahí, lo único que se ejecuta son los callbacks, y solo aquellos cuyas entradas hayan cambiado.

Las consecuencias prácticas son grandes:

| | Streamlit | Dash |
|---|---|---|
| Ejecución del script | En cada interacción | Una vez, al arrancar |
| Qué se ejecuta al interactuar | Todo | Solo los callbacks afectados |
| Cargar un modelo grande | Necesita `@st.cache_resource` | Basta con ponerlo a nivel de módulo |
| Actualizar solo un gráfico | Difícil (todo se recalcula) | Natural (un callback, una salida) |
| Curva de aprendizaje | Muy suave | Moderada |
| Control del diseño | Limitado | Total |

El código que carga datos o modelos va **a nivel de módulo**, fuera de cualquier función, y se queda en memoria del proceso:

```python
import pandas as pd

DF = pd.read_csv("ventas.csv")     # se ejecuta UNA vez, al arrancar
MODELO = joblib.load("modelo.joblib")

app = Dash(__name__)
# ... layout y callbacks usan DF y MODELO libremente
```

Si vienes de Streamlit, la tentación de envolver esto en un decorador de caché es fuerte. No hace falta: en Dash no hay nada que cachear porque nada se repite.

## 0.4 Qué gana y qué pierde frente a Streamlit

**Dash gana en:**

- **Control del diseño.** El layout es HTML real; puedes colocar cualquier cosa donde quieras.
- **Rendimiento con interfaces complejas.** Solo se recalcula lo que depende de lo que ha cambiado.
- **Aplicaciones grandes.** El modelo de callbacks escala mejor cuando hay decenas de controles.
- **Interacción entre gráficos.** Hacer clic en un gráfico para filtrar otro es natural.
- **Producción.** Al ser una aplicación Flask estándar, encaja en cualquier infraestructura.

**Dash pierde en:**

- **Verbosidad.** Lo que en Streamlit son 3 líneas, en Dash son 15.
- **Curva de aprendizaje.** Hay que entender el sistema de callbacks.
- **Prototipado rápido.** Para probar una idea en cinco minutos, Streamlit gana.
- **Despliegue gratuito sencillo.** Streamlit tiene Community Cloud; Dash necesita Render, Docker o similar (módulo 9).

**Criterio práctico:** si es una demo que vas a enseñar y desechar, Streamlit. Si es un cuadro de mando que alguien va a usar todos los días durante dos años, Dash.

## 0.5 Dash frente a Gradio

Curiosamente, Dash se parece más a Gradio que a Streamlit en su filosofía: ambos declaran componentes y luego conectan eventos explícitamente. Las diferencias son de propósito:

- **Gradio** está optimizado para envolver un modelo: entra un dato, sale una predicción. Sus componentes multimedia son excelentes.
- **Dash** está optimizado para explorar datos: filtros, gráficos enlazados, tablas.

---

# Módulo 1. Instalación y primera aplicación

## 1.1 Instalación

Se recomienda un entorno virtual, por las razones habituales (aislar dependencias, y poder generar un `requirements.txt` limpio para el despliegue):

```bash
mkdir mi-dashboard && cd mi-dashboard
python -m venv venv

# Linux / macOS
source venv/bin/activate
# Windows PowerShell
venv\Scripts\Activate.ps1
```

Con el entorno activado:

```bash
pip install dash
```

Dash arrastra Plotly y Flask automáticamente. Para el resto del curso instalaremos también:

```bash
pip install "dash[ag-grid]" dash-bootstrap-components pandas gunicorn
```

Qué es cada cosa:

- **`dash[ag-grid]`** añade AG Grid, el componente de tabla moderno. Es necesario porque **`dash_table.DataTable` está deprecado en Dash 4** (más sobre esto en el módulo 7).
- **`dash-bootstrap-components`** aporta un sistema de rejilla y componentes con estilo, y ahorra escribir CSS.
- **`gunicorn`** es el servidor de producción. Se instala ahora para que esté en `requirements.txt` desde el principio.

Comprobación:

```python
import dash
print(dash.__version__)      # 4.4.1 o superior
```

## 1.2 La aplicación mínima

Crea `app.py`:

```python
from dash import Dash, html

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Mi primera aplicación Dash"),
    html.P("Esto es un párrafo."),
])

if __name__ == "__main__":
    app.run(debug=True)
```

Ejecuta:

```bash
python app.py
```

A diferencia de Streamlit, aquí **sí se usa `python`**, porque el script arranca el servidor él mismo.

Verás:

```
Dash is running on http://127.0.0.1:8050/
```

Abre esa dirección en el navegador.

## 1.3 Anatomía de lo que acaba de pasar

```python
app = Dash(__name__)
```

Crea la aplicación. `__name__` le sirve a Flask para localizar los recursos estáticos (la carpeta `assets/`, que veremos en el módulo 3).

```python
app.layout = html.Div([...])
```

Define el árbol de componentes. `html.Div` genera un `<div>`; su primer argumento posicional son los hijos.

```python
app.run(debug=True)
```

Arranca el servidor de desarrollo. `debug=True` activa dos cosas muy útiles:

- **Recarga automática**: al guardar el fichero, la aplicación se reinicia sola.
- **Herramientas de depuración**: un botón azul abajo a la derecha que muestra los errores y permite inspeccionar el grafo de callbacks.

> **Aviso sobre `app.run`.** En Dash 2.x el método se llamaba `app.run_server()`. Fue deprecado y **eliminado en Dash 3**. Si sigues un tutorial antiguo y ves `run_server`, sustitúyelo por `run`.

## 1.4 Primera aplicación interactiva

Ahora con un callback:

```python
from dash import Dash, html, dcc, callback, Input, Output

app = Dash(__name__)

app.layout = html.Div([
    html.H2("Conversor de temperatura"),

    dcc.Slider(
        id="slider-celsius",
        min=-20, max=50, step=1, value=20,
        marks={-20: "-20°C", 0: "0°C", 25: "25°C", 50: "50°C"},
    ),

    html.Div(id="resultado", style={"fontSize": 24, "marginTop": 20}),
])


@callback(
    Output("resultado", "children"),
    Input("slider-celsius", "value"),
)
def convertir(celsius):
    fahrenheit = celsius * 9 / 5 + 32
    return f"{celsius} °C = {fahrenheit:.1f} °F"


if __name__ == "__main__":
    app.run(debug=True)
```

Tres detalles que conviene fijar desde ya:

1. **Los `id` son obligatorios** para cualquier componente que participe en un callback, y deben ser únicos en toda la aplicación.
2. **El callback se define fuera del layout**, normalmente después.
3. **La función se ejecuta también al cargar la página**, no solo al mover el slider. Dash dispara todos los callbacks una vez al inicio para poblar la interfaz. Si no quieres ese comportamiento, se desactiva con `prevent_initial_call=True`.

## 1.5 Opciones de arranque

```python
app.run(
    debug=True,           # recarga automática + herramientas de depuración
    host="127.0.0.1",     # "0.0.0.0" para exponer en la red local o en Docker
    port=8050,
    dev_tools_hot_reload=True,
)
```

## 1.6 Problemas frecuentes al empezar

| Síntoma | Causa | Solución |
|---|---|---|
| `AttributeError: 'Dash' object has no attribute 'run_server'` | Tutorial antiguo | Usa `app.run()` |
| `Address already in use` | Otra instancia corriendo | Ciérrala, o usa `port=8051` |
| `Duplicate callback outputs` | Dos callbacks escriben en la misma salida | Ver módulo 5.8 |
| `nonexistent object was used in an Input` | El `id` no existe en el layout | Revisa la ortografía del `id` |
| La página sale en blanco | Error de Python al construir el layout | Mira la terminal |
| Los cambios no se ven | `debug=False` | Actívalo en desarrollo |

---

# Módulo 2. Ejecución desde Google Colab

Si trabajas en Colab en lugar de en tu ordenador, hay un detalle: Colab se ejecuta en un servidor de Google, y su `localhost` no es el tuyo.

## 2.1 El método nativo (recomendado)

A diferencia de Streamlit, **Dash trae soporte integrado para Colab y Jupyter**. No hace falta ngrok:

```python
!pip install -q dash dash-bootstrap-components

from dash import Dash, html, dcc, callback, Input, Output

app = Dash(__name__)
app.layout = html.Div([
    dcc.Slider(id="s", min=0, max=10, value=5),
    html.Div(id="salida"),
])

@callback(Output("salida", "children"), Input("s", "value"))
def f(v):
    return f"Valor: {v}"

# jupyter_mode="inline" incrusta la app en la salida de la celda
app.run(jupyter_mode="inline", debug=True)
```

Valores posibles de `jupyter_mode`:

| Valor | Efecto |
|---|---|
| `"inline"` | Incrusta la aplicación en la salida de la celda |
| `"external"` | Muestra un enlace para abrirla en otra pestaña |
| `"tab"` | Abre una pestaña nueva automáticamente |
| `"jupyterlab"` | Panel lateral en JupyterLab |

Esta capacidad venía antes de la librería `jupyter-dash`, que **ya no hace falta instalar**: se integró en Dash 2.11.

## 2.2 Si necesitas una URL pública

El método anterior solo funciona para ti, en tu sesión. Para enseñárselo a alguien, la opción es ngrok, igual que con Streamlit:

```python
!pip install -q pyngrok

from pyngrok import ngrok
import threading

ngrok.set_auth_token("TU_TOKEN")   # dashboard.ngrok.com/get-started/your-authtoken
ngrok.kill()

def arrancar():
    app.run(host="0.0.0.0", port=8050)

threading.Thread(target=arrancar, daemon=True).start()

url = ngrok.connect(8050)
print("Aplicación pública en:", url.public_url)
```

Recuerda que la URL de ngrok **muere al cerrar Colab**. Para publicar de verdad, módulo 9.

---

# Módulo 3. El layout: construir la página con Python

## 3.1 El árbol de componentes

El layout es una estructura anidada de objetos Python. Cada componente se traduce a un elemento HTML:

```python
app.layout = html.Div([
    html.H1("Título"),
    html.Div([
        html.P("Primer párrafo"),
        html.P("Segundo párrafo"),
    ]),
])
```

Genera:

```html
<div>
  <h1>Título</h1>
  <div>
    <p>Primer párrafo</p>
    <p>Segundo párrafo</p>
  </div>
</div>
```

La correspondencia es directa: `html.H1` → `<h1>`, `html.Table` → `<table>`, y así con todas las etiquetas HTML.


## 3.2 La propiedad `children`

El primer argumento posicional de cualquier componente es `children`, y acepta:

```python
html.Div("Un texto")                          # una cadena
html.Div(42)                                   # un número
html.Div(html.P("Un componente"))              # otro componente
html.Div([html.P("Uno"), html.P("Dos")])       # una lista de componentes
html.Div(children=["Mezcla", html.B("negrita"), "de cosas"])
```

Escribirlo de forma explícita (`children=[...]`) es más legible en layouts grandes.

## 3.3 Estilos

Se aplican con el argumento `style`, que recibe un diccionario. **Atención a dos diferencias con el CSS normal:**

```python
html.Div(
    "Contenido",
    style={
        "backgroundColor": "#f1f5f9",   # camelCase, NO 'background-color'
        "padding": "20px",               # las unidades van como cadena
        "borderRadius": "8px",
        "fontSize": 18,                  # los números se interpretan como px
    },
)
```

1. Las propiedades van en **camelCase**, no con guiones.
2. Los valores con unidades son **cadenas**; un número suelto se interpreta como píxeles.


## 3.4 Clases CSS y la carpeta `assets`

Para algo más que estilos puntuales, usa clases:

```python
html.Div("Contenido", className="mi-tarjeta")
```

Fíjate en que es `className`, no `class` (que es palabra reservada en Python).

Dash carga **automáticamente** cualquier `.css` o `.js` que encuentre en una carpeta llamada `assets/` junto a `app.py`:

```
mi-dashboard/
├── app.py
└── assets/
    ├── estilos.css
    └── favicon.ico
```

`assets/estilos.css`:

```css
.mi-tarjeta {
    background: white;
    border-radius: 8px;
    padding: 20px;
    box-shadow: 0 1px 3px rgba(0,0,0,.1);
}
```

No hay que declarar nada: Dash detecta la carpeta y sirve su contenido. El `favicon.ico` también se recoge solo.

## 3.5 Rejilla con Bootstrap

Escribir CSS para colocar cosas es tedioso. `dash-bootstrap-components` resuelve el 90 % de los casos:

```python
import dash_bootstrap_components as dbc

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.Div("Izquierda"), width=4),
        dbc.Col(html.Div("Centro"), width=4),
        dbc.Col(html.Div("Derecha"), width=4),
    ]),
], fluid=True)
```

El sistema de rejilla de Bootstrap divide el ancho en **12 columnas**. `width=4` ocupa un tercio; `width=6`, la mitad.

Para diseño adaptable, se especifica el ancho por tamaño de pantalla:

```python
dbc.Col(contenido, xs=12, md=6, lg=4)
# móvil: ancho completo · tableta: mitad · escritorio: un tercio
```

Componentes de Bootstrap más útiles:

```python
dbc.Card(dbc.CardBody([html.H5("Título"), html.P("Texto")]))
dbc.Button("Pulsar", color="primary", size="sm")
dbc.Alert("Aviso importante", color="warning")
dbc.Tabs([dbc.Tab(label="Uno"), dbc.Tab(label="Dos")])
dbc.Spinner(children=[...])
```

Y hay temas alternativos:

```python
external_stylesheets=[dbc.themes.BOOTSTRAP]   # el clásico
external_stylesheets=[dbc.themes.FLATLY]      # plano y moderno
external_stylesheets=[dbc.themes.DARKLY]      # oscuro
```

La lista completa está en `dash-bootstrap-components.opensource.faculty.ai/docs/themes/`.

---

# Módulo 4. Componentes

## 4.1 Los tres paquetes

| Paquete | Qué contiene |
|---|---|
| `dash.html` | Los elementos HTML (`Div`, `H1`, `P`, `Table`, `Img`...) |
| `dash.dcc` | Los componentes interactivos (*Dash Core Components*) |
| `dash_bootstrap_components` | Componentes con estilo Bootstrap |

## 4.2 Componentes de entrada (`dcc`)

### Desplegables

```python
dcc.Dropdown(
    id="mi-dropdown",
    options=["Madrid", "Barcelona", "Valencia"],   # forma simple
    value="Madrid",              # seleccionado por defecto
    multi=False,                 # True permite selección múltiple
    clearable=True,              # muestra la X para limpiar
    searchable=True,
    placeholder="Elige una ciudad",
)
```

Cuando la etiqueta visible debe diferir del valor interno, se usa la forma completa:

```python
options=[
    {"label": "Madrid (capital)", "value": "MAD"},
    {"label": "Barcelona", "value": "BCN"},
]
```

Con `multi=True`, la propiedad `value` pasa a ser una **lista**.

### Deslizadores

```python
dcc.Slider(
    id="mi-slider",
    min=0, max=100, step=5, value=50,
    marks={0: "0", 50: "50", 100: "100"},
    tooltip={"placement": "bottom", "always_visible": True},
)

dcc.RangeSlider(          # dos manijas: value es una lista de dos
    id="rango", min=0, max=100, value=[20, 80],
)
```

### Texto y números

```python
dcc.Input(id="texto", type="text", placeholder="Escribe...", debounce=True)
dcc.Input(id="numero", type="number", min=0, max=100, value=50)
dcc.Textarea(id="largo", style={"width": "100%", "height": 120})
```

> **`debounce=True` es importante.** Sin él, el callback se dispara con **cada tecla pulsada**. Con él, espera a que el usuario pulse Enter o salga del campo. En un campo conectado a una consulta pesada, la diferencia es enorme.

### Selección

```python
dcc.RadioItems(id="radio", options=["A", "B", "C"], value="A", inline=True)
dcc.Checklist(id="check", options=["X", "Y", "Z"], value=["X"])
```

`dcc.Checklist` devuelve siempre una **lista**.

### Fechas

```python
dcc.DatePickerSingle(id="fecha", date="2026-01-01", display_format="DD/MM/YYYY")

dcc.DatePickerRange(
    id="periodo",
    start_date="2026-01-01", end_date="2026-12-31",
    min_date_allowed="2020-01-01", max_date_allowed="2030-12-31",
    display_format="DD/MM/YYYY",
)
```

`DatePickerRange` tiene **dos propiedades** relevantes: `start_date` y `end_date`. Un callback que dependa del periodo necesita las dos como `Input`.

### Subida de ficheros

```python
dcc.Upload(
    id="subida",
    children=html.Div(["Arrastra un fichero o ", html.A("selecciónalo")]),
    style={
        "width": "100%", "height": "60px", "lineHeight": "60px",
        "borderWidth": "1px", "borderStyle": "dashed",
        "borderRadius": "5px", "textAlign": "center",
    },
    multiple=False,
)
```

El contenido llega **codificado en base64**, y hay que decodificarlo:

```python
import base64, io
import pandas as pd

@callback(
    Output("salida", "children"),
    Input("subida", "contents"),
    State("subida", "filename"),
)
def procesar(contenido, nombre):
    if contenido is None:
        return "Sin fichero"

    tipo, cadena = contenido.split(",")
    decodificado = base64.b64decode(cadena)

    try:
        df = pd.read_csv(io.StringIO(decodificado.decode("utf-8")))
    except Exception as e:
        return f"Error al leer el fichero: {e}"

    return f"{nombre}: {len(df)} filas"
```

Es bastante más verboso que el `st.file_uploader` de Streamlit. Es el precio del control.

### Otros

```python
dcc.Graph(id="grafico", figure=fig)           # gráfico Plotly (módulo 6)
dcc.Store(id="almacen")                        # datos en el navegador (módulo 7)
dcc.Download(id="descarga")                    # descargar ficheros
dcc.Interval(id="reloj", interval=5000)        # dispara cada 5 segundos
dcc.Location(id="url")                         # lee y cambia la URL
dcc.Loading(children=[...])                    # indicador de carga
dcc.Tabs([dcc.Tab(label="Uno"), dcc.Tab(label="Dos")])
dcc.Markdown("Texto con **formato**")
```

## 4.3 Indicadores de carga

Para que el usuario sepa que algo está pasando, se envuelve el componente lento:

```python
dcc.Loading(
    id="cargando",
    type="circle",         # "graph", "cube", "dot", "default"
    children=[dcc.Graph(id="grafico-lento")],
)
```

Dash muestra la animación automáticamente mientras el callback que alimenta a `grafico-lento` está en curso. No hay que gestionarlo a mano.

---

# Módulo 5. Callbacks: el corazón de Dash

Este es el módulo central. Todo lo que hace una aplicación de Dash pasa por aquí.

## 5.1 Anatomía

```python
from dash import callback, Input, Output

@callback(
    Output("id-destino", "propiedad"),
    Input("id-origen", "propiedad"),
)
def mi_funcion(valor_de_entrada):
    resultado = hacer_algo(valor_de_entrada)
    return resultado
```

Las tres reglas que gobiernan todo:

1. **Cada `Input` se convierte en un argumento** de la función, en el mismo orden en que se declara.
2. **Cada `Output` recibe un valor devuelto**, en el mismo orden.
3. **El callback se dispara cuando cambia cualquiera de sus `Input`.**

El primer argumento de `Output` e `Input` es el `id` del componente; el segundo, la propiedad concreta que se observa o se modifica.

## 5.2 Qué propiedad usar

Cada tipo de componente expone propiedades distintas. Las más habituales:

| Componente | Propiedad de entrada | Propiedad de salida |
|---|---|---|
| `dcc.Dropdown` | `value` | `options`, `value` |
| `dcc.Slider` | `value` | `value`, `max` |
| `dcc.Input` | `value` | `value` |
| `dcc.Checklist` | `value` (lista) | `options`, `value` |
| `dcc.DatePickerRange` | `start_date`, `end_date` | idem |
| `dcc.Graph` | `clickData`, `selectedData`, `hoverData` | `figure` |
| `dcc.Upload` | `contents`, `filename` | — |
| `html.Button` | `n_clicks` | — |
| `html.Div` | — | `children` |
| `dag.AgGrid` | `selectedRows`, `cellClicked` | `rowData`, `columnDefs` |

`children` es la propiedad universal de salida: sirve para meter texto o componentes dentro de cualquier elemento HTML.


## 5.3 Varias entradas

```python
@callback(
    Output("resultado", "children"),
    Input("precio", "value"),
    Input("cantidad", "value"),
    Input("descuento", "value"),
)
def calcular(precio, cantidad, descuento):
    total = precio * cantidad * (1 - descuento / 100)
    return f"Total: {total:.2f} €"
```

El callback se dispara cuando cambia **cualquiera** de los tres. El orden de los argumentos de la función debe coincidir con el de los `Input`.

## 5.4 Varias salidas

```python
@callback(
    Output("kpi-total", "children"),
    Output("kpi-media", "children"),
    Output("kpi-maximo", "children"),
    Input("filtro", "value"),
)
def actualizar_kpis(filtro):
    d = DF[DF["categoria"] == filtro]
    return f"{d['ventas'].sum():,.0f}", f"{d['ventas'].mean():,.0f}", f"{d['ventas'].max():,.0f}"
```

La función debe devolver **exactamente tantos valores como `Output` haya declarados**, en el mismo orden. Es el error más común al empezar.

## 5.5 `State`: leer sin disparar

A veces necesitas un valor pero no quieres que su cambio dispare el callback. Para eso está `State`:

```python
from dash import State

@callback(
    Output("salida", "children"),
    Input("boton", "n_clicks"),          # esto dispara
    State("nombre", "value"),            # esto solo se lee
    State("email", "value"),
    prevent_initial_call=True,
)
def enviar(n_clicks, nombre, email):
    return f"Enviado: {nombre} ({email})"
```

Aquí el usuario escribe nombre y correo sin que pase nada, y el callback solo se ejecuta al pulsar el botón. Es el patrón equivalente a `st.form` de Streamlit.

Los argumentos de la función siguen este orden: **primero todos los `Input`, después todos los `State`**.

## 5.6 Botones y `n_clicks`

Un botón no tiene un valor propio: expone `n_clicks`, que empieza en `None` y se incrementa con cada pulsación.

```python
html.Button("Calcular", id="btn", n_clicks=0)

@callback(
    Output("salida", "children"),
    Input("btn", "n_clicks"),
    prevent_initial_call=True,
)
def calcular(n):
    return f"Has pulsado {n} veces"
```

`prevent_initial_call=True` evita que el callback se ejecute al cargar la página, cuando `n_clicks` todavía es `None`.

Sin ese argumento, hay que protegerse a mano:

```python
def calcular(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    ...
```

## 5.7 `PreventUpdate` y `no_update`

Dos formas de decirle a Dash «no cambies nada»:

```python
from dash.exceptions import PreventUpdate
import dash

# Opción A: cancelar el callback entero
@callback(Output("salida", "children"), Input("entrada", "value"))
def f(valor):
    if not valor:
        raise PreventUpdate       # ninguna salida se actualiza
    return procesar(valor)

# Opción B: actualizar unas salidas sí y otras no
@callback(
    Output("a", "children"),
    Output("b", "children"),
    Input("entrada", "value"),
)
def g(valor):
    if not valor:
        return "sin datos", dash.no_update   # 'b' se queda como estaba
    return calcular_a(valor), calcular_b(valor)
```

## 5.8 Callbacks encadenados

La salida de un callback puede ser la entrada de otro. Dash construye el grafo de dependencias y los ejecuta en orden:

```python
@callback(Output("provincias", "options"), Input("pais", "value"))
def cargar_provincias(pais):
    return sorted(DF[DF["pais"] == pais]["provincia"].unique())


@callback(Output("ciudades", "options"), Input("provincias", "value"))
def cargar_ciudades(provincia):
    return sorted(DF[DF["provincia"] == provincia]["ciudad"].unique())
```

Es el patrón de **desplegables dependientes**: elegir un país actualiza la lista de provincias, y eso actualiza la de ciudades.

Un detalle práctico: al cambiar las `options` conviene resetear también el `value`, o quedará seleccionado un valor que ya no está en la lista:

```python
@callback(
    Output("provincias", "options"),
    Output("provincias", "value"),
    Input("pais", "value"),
)
def cargar_provincias(pais):
    opciones = sorted(DF[DF["pais"] == pais]["provincia"].unique())
    return opciones, None      # None limpia la selección
```


## 5.9 Salidas duplicadas

Por defecto, **dos callbacks no pueden escribir en la misma salida**. Dash lanza `DuplicateCallbackOutput` al arrancar.

Cuando lo necesites de verdad (por ejemplo, un valor que puede cambiar desde un botón o desde un desplegable), se declara explícitamente:

```python
@callback(
    Output("resultado", "children", allow_duplicate=True),
    Input("boton-reset", "n_clicks"),
    prevent_initial_call=True,     # obligatorio con allow_duplicate
)
def resetear(n):
    return "Reiniciado"
```

`allow_duplicate=True` **exige** `prevent_initial_call=True`.


## 5.10 Saber qué disparó el callback: `ctx`

Cuando un callback tiene varias entradas y necesitas saber cuál cambió:

```python
from dash import ctx

@callback(
    Output("salida", "children"),
    Input("btn-guardar", "n_clicks"),
    Input("btn-borrar", "n_clicks"),
    prevent_initial_call=True,
)
def gestionar(guardar, borrar):
    if ctx.triggered_id == "btn-guardar":
        return "Guardado"
    elif ctx.triggered_id == "btn-borrar":
        return "Borrado"
    return dash.no_update
```

`ctx.triggered_id` devuelve el `id` del componente que disparó el callback. Es la forma limpia de manejar varios botones con un solo callback.

## 5.11 Interacción con gráficos

Los gráficos exponen propiedades que se actualizan al interactuar con ellos:

```python
@callback(
    Output("detalle", "children"),
    Input("mi-grafico", "clickData"),
)
def mostrar_detalle(clic):
    if clic is None:
        return "Haz clic en un punto"

    punto = clic["points"][0]
    return f"x={punto['x']}, y={punto['y']}"
```

Las tres propiedades disponibles:

| Propiedad | Cuándo se actualiza |
|---|---|
| `clickData` | Al hacer clic en un punto |
| `hoverData` | Al pasar el ratón por encima |
| `selectedData` | Al seleccionar una región con el lazo o el rectángulo |

La estructura del diccionario varía según el tipo de gráfico. La forma fiable de averiguarla es imprimirla:

```python
@callback(Output("depuracion", "children"), Input("grafico", "clickData"))
def ver(clic):
    import json
    return json.dumps(clic, indent=2)
```

Este es el mecanismo que permite el **filtrado cruzado**: hacer clic en una barra de un gráfico para filtrar una tabla. Lo usaremos en el módulo 8.

## 5.12 Callbacks en segundo plano

Para tareas que tardan más de unos segundos (entrenar un modelo, una consulta pesada), un callback normal bloquea al usuario y puede agotar el tiempo de espera del servidor. La solución son los *background callbacks*:

```python
import diskcache
from dash import DiskcacheManager

cache = diskcache.Cache("./cache")
gestor = DiskcacheManager(cache)

app = Dash(__name__, background_callback_manager=gestor)


@callback(
    Output("resultado", "children"),
    Input("btn", "n_clicks"),
    background=True,
    running=[
        (Output("btn", "disabled"), True, False),      # desactiva el botón mientras corre
        (Output("progreso", "style"), {"display": "block"}, {"display": "none"}),
    ],
    progress=[Output("barra", "value"), Output("barra", "max")],
    prevent_initial_call=True,
)
def tarea_larga(set_progress, n):
    total = 100
    for i in range(total):
        time.sleep(0.1)
        set_progress((str(i + 1), str(total)))
    return "Completado"
```

`DiskcacheManager` sirve para desarrollo. En producción con varios procesos se usa `CeleryManager`, que requiere Redis o RabbitMQ.

## 5.13 Callbacks del lado del cliente

Cuando la operación es trivial (formatear un texto, mostrar u ocultar algo), enviar una petición al servidor es un desperdicio. Se puede ejecutar JavaScript directamente en el navegador:

```python
from dash import clientside_callback

clientside_callback(
    """
    function(valor) {
        return valor ? "Activado" : "Desactivado";
    }
    """,
    Output("estado", "children"),
    Input("interruptor", "value"),
)
```

La respuesta es instantánea porque no hay viaje al servidor. Úsalo solo para lógica sencilla que no necesite datos de Python.

---

# Módulo 6. Gráficos con Plotly

Dash usa Plotly, así que el gráfico es un objeto que se asigna a la propiedad `figure` de un `dcc.Graph`.

## 6.1 Plotly Express: la vía rápida

```python
import plotly.express as px

fig = px.line(df, x="fecha", y="ventas", color="region")
fig = px.bar(df, x="categoria", y="total")
fig = px.scatter(df, x="precio", y="unidades", size="beneficio", color="canal")
fig = px.pie(df, names="categoria", values="ingresos", hole=0.5)
fig = px.histogram(df, x="importe", nbins=30)
fig = px.box(df, x="region", y="margen")
fig = px.treemap(df, path=["region", "categoria"], values="ingresos")
fig = px.imshow(matriz_correlacion, text_auto=True)
```

Plotly Express funciona directamente sobre un DataFrame: le indicas qué columna va en cada eje y él agrupa y colorea.

## 6.2 Graph Objects: control total

Cuando necesitas combinar tipos de gráfico o afinar detalles:

```python
import plotly.graph_objects as go

fig = go.Figure()
fig.add_bar(x=meses, y=ingresos, name="Ingresos", marker_color="#2563eb")
fig.add_scatter(x=meses, y=beneficio, name="Beneficio",
                mode="lines+markers", line=dict(color="#16a34a", width=3))
fig.update_layout(
    template="plotly_white",
    margin=dict(l=10, r=10, t=30, b=10),
    legend=dict(orientation="h", y=1.1),
    hovermode="x unified",
)
```

## 6.3 Ajustes que casi siempre querrás

```python
fig.update_layout(
    template="plotly_white",              # fondo limpio (o "plotly_dark")
    margin=dict(l=10, r=10, t=30, b=10),  # aprovechar el espacio
    showlegend=True,
    legend=dict(orientation="h", y=1.1, x=0),
    hovermode="x unified",                # un solo tooltip para todas las series
    xaxis_title=None,                     # quitar títulos redundantes
    yaxis_title=None,
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(tickformat=",.0f")
fig.update_traces(textposition="outside")
```

En un cuadro de mando, los márgenes por defecto de Plotly desperdician mucho espacio. Reducirlos es de las cosas que más mejoran el aspecto.

## 6.4 Colores coherentes

Un error frecuente: que la misma categoría salga de un color en un gráfico y de otro en el siguiente. Se evita fijando el mapa de colores:

```python
COLORES = ["#2563eb", "#16a34a", "#ea580c", "#7c3aed", "#dc2626"]
MAPA = {cat: COLORES[i % len(COLORES)] for i, cat in enumerate(CATEGORIAS)}

fig = px.bar(df, x="categoria", y="total",
             color="categoria", color_discrete_map=MAPA)
```

## 6.5 Configuración del componente

```python
dcc.Graph(
    id="mi-grafico",
    figure=fig,
    config={
        "displayModeBar": False,      # oculta la barra de herramientas
        "displaylogo": False,         # quita el logo de Plotly
        "scrollZoom": False,
        "locale": "es",
    },
    style={"height": "350px"},
)
```

En un cuadro de mando, `displayModeBar: False` suele ser lo adecuado: la barra de herramientas de Plotly distrae y rara vez se usa.

## 6.6 Gráficos vacíos

Cuando los filtros no devuelven datos, un `px.bar` sobre un DataFrame vacío da un gráfico feo o un error. Conviene una figura de reserva:

```python
def figura_vacia(mensaje="Sin datos para los filtros seleccionados"):
    fig = go.Figure()
    fig.add_annotation(
        text=mensaje, xref="paper", yref="paper", x=0.5, y=0.5,
        showarrow=False, font=dict(size=14, color="#94a3b8"),
    )
    fig.update_layout(
        template="plotly_white",
        xaxis=dict(visible=False), yaxis=dict(visible=False),
    )
    return fig


@callback(Output("grafico", "figure"), Input("filtro", "value"))
def actualizar(filtro):
    d = filtrar(DF, filtro)
    if d.empty:
        return figura_vacia()
    return px.bar(d, x="categoria", y="total")
```

---

# Módulo 7. Tablas, estado y rendimiento

## 7.1 Tablas: AG Grid, no DataTable

**Aviso importante de versión.** `dash_table.DataTable`, que aparece en prácticamente todos los tutoriales, **está deprecado en Dash 4**. Al usarlo se obtiene:

```
DeprecationWarning: The dash_table.DataTable will be removed from the builtin
dash components in a future major version. We recommend using dash-ag-grid as
a replacement. Install with `pip install dash[ag-grid]`.
```

El sustituto es **AG Grid**, que además es bastante mejor: más rápido con muchas filas, con filtros por columna, agrupación y exportación.

```python
import dash_ag_grid as dag

dag.AgGrid(
    id="tabla",
    rowData=df.to_dict("records"),
    columnDefs=[
        {"field": "producto", "headerName": "Producto", "flex": 1},
        {"field": "region", "headerName": "Región", "width": 120},
        {"field": "ingresos", "headerName": "Ingresos",
         "type": "numericColumn",
         "valueFormatter": {"function": "d3.format(',.2f')(params.value) + ' €'"}},
    ],
    defaultColDef={"sortable": True, "filter": True, "resizable": True},
    dashGridOptions={"pagination": True, "paginationPageSize": 20},
    style={"height": "500px"},
    className="ag-theme-alpine",
)
```

Puntos a recordar:

- **`rowData`** son los datos, como lista de diccionarios.
- **`columnDefs`** define las columnas; `field` debe coincidir con la clave del diccionario.
- **`flex: 1`** hace que la columna ocupe el espacio sobrante; `width` la fija.
- **`valueFormatter`** admite una función JavaScript para formatear (aquí, formato europeo con símbolo de euro).
- **`className`** elige el tema; debe llevar el prefijo `ag-theme-`.

Para leer la selección del usuario:

```python
dag.AgGrid(
    id="tabla",
    dashGridOptions={"rowSelection": "multiple"},
    ...
)

@callback(Output("detalle", "children"), Input("tabla", "selectedRows"))
def mostrar(filas):
    if not filas:
        return "Selecciona una fila"
    return f"{len(filas)} filas seleccionadas"
```

## 7.2 `dcc.Store`: compartir datos entre callbacks

`dcc.Store` guarda datos **en el navegador del usuario**:

```python
dcc.Store(id="mi-almacen", storage_type="memory")
```

| `storage_type` | Duración |
|---|---|
| `"memory"` | Se pierde al recargar la página |
| `"session"` | Sobrevive a recargas, se pierde al cerrar la pestaña |
| `"local"` | Persiste indefinidamente |

Es útil para el estado de la aplicación: qué elemento está seleccionado, preferencias del usuario, un contador.

### El antipatrón que debes evitar

Aquí hay una trampa en la que cae mucha gente, yo incluido al escribir la primera versión del ejemplo de este curso.

La idea parece razonable: como varios callbacks necesitan los mismos datos filtrados, filtramos una vez, lo guardamos en un `Store`, y los demás leen de ahí.

```python
# MAL con volúmenes grandes
@callback(Output("almacen", "data"), Input("filtro", "value"))
def filtrar_una_vez(filtro):
    return DF[DF["categoria"] == filtro].to_dict("records")


@callback(Output("grafico", "figure"), Input("almacen", "data"))
def dibujar(datos):
    return px.bar(pd.DataFrame(datos), x="mes", y="total")
```

El problema es que `dcc.Store` vive **en el navegador**. Eso significa que los datos se serializan a JSON, viajan al cliente, y vuelven al servidor en cada callback que los consume.

En el ejemplo del módulo 8, con 23.307 filas, eso son **8 MB de JSON en cada interacción**. El resultado es una aplicación lentísima, y con volúmenes algo mayores, errores 500 por tamaño de petición.

### La alternativa correcta

Como en Dash el proceso de Python **persiste entre peticiones**, el DataFrame ya está en memoria del servidor. Basta con que cada callback filtre por su cuenta, y memoizar para no repetir el trabajo:

```python
from functools import lru_cache

@lru_cache(maxsize=64)
def _filtrar_memo(desde, hasta, regiones, categorias):
    return filtrar(DF, desde, hasta, list(regiones), list(categorias))


def datos(desde, hasta, regiones, categorias):
    """Punto de entrada único. Devuelve una COPIA."""
    return _filtrar_memo(
        desde, hasta, tuple(regiones or []), tuple(categorias or [])
    ).copy()
```

Dos detalles imprescindibles:

- **`lru_cache` exige argumentos hasheables**, y las listas no lo son. De ahí la conversión a tuplas.
- **Devolver una copia.** `lru_cache` entrega siempre el mismo objeto; si un callback lo modificara, corrompería la caché para todos los demás.

El resultado en el ejemplo real: el cuerpo de cada petición pasa de **8 MB a 300 bytes**, y el callback de los KPI baja a **7 ms**.

**Regla:** usa `dcc.Store` para estado ligero (selecciones, preferencias, identificadores). Para datos tabulares, filtra en el servidor.


## 7.3 Caché entre usuarios con Flask-Caching

`lru_cache` vive en el proceso. Si despliegas con varios *workers* de gunicorn, cada uno tendrá su propia caché. Para compartirla:

```python
from flask_caching import Cache

cache = Cache(app.server, config={
    "CACHE_TYPE": "FileSystemCache",
    "CACHE_DIR": "cache",
    "CACHE_DEFAULT_TIMEOUT": 3600,
})


@cache.memoize()
def consulta_pesada(parametro):
    return pd.read_sql(f"SELECT ... WHERE x = {parametro}", conexion)
```

Con `CACHE_TYPE: "RedisCache"` la caché se comparte entre procesos y máquinas.

## 7.4 Actualización automática

```python
dcc.Interval(id="reloj", interval=30_000, n_intervals=0)   # cada 30 segundos

@callback(Output("kpi", "children"), Input("reloj", "n_intervals"))
def refrescar(n):
    return leer_valor_actual()
```

Útil para paneles de monitorización. Cuidado con intervalos cortos: cada disparo es una petición al servidor por cada usuario conectado.

## 7.5 Aplicaciones multipágina

```
mi-dashboard/
├── app.py
└── pages/
    ├── inicio.py
    ├── analisis.py
    └── detalle.py
```

`app.py`:

```python
from dash import Dash, html, dcc, page_container
import dash_bootstrap_components as dbc

app = Dash(__name__, use_pages=True, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.NavbarSimple([
        dbc.NavItem(dbc.NavLink("Inicio", href="/")),
        dbc.NavItem(dbc.NavLink("Análisis", href="/analisis")),
    ], brand="Mi cuadro de mando"),

    page_container,        # aquí se inyecta la página activa
], fluid=True)

server = app.server

if __name__ == "__main__":
    app.run(debug=True)
```

`pages/analisis.py`:

```python
import dash
from dash import html, dcc, callback, Input, Output

dash.register_page(__name__, path="/analisis", name="Análisis")

layout = html.Div([          # se llama 'layout', no 'app.layout'
    html.H2("Análisis"),
    dcc.Graph(id="grafico-analisis"),
])


@callback(Output("grafico-analisis", "figure"), Input("filtro", "value"))
def actualizar(f):
    ...
```

Cada página registra su ruta con `dash.register_page` y expone una variable `layout`. Dash monta la navegación.


## 7.6 Rendimiento: lista de comprobación

- **Carga los datos a nivel de módulo**, no dentro de callbacks.
- **Memoiza el filtrado** con `lru_cache` o `flask_caching`.
- **No metas DataFrames grandes en `dcc.Store`.**
- **Agrega antes de dibujar.** Un gráfico con 100.000 puntos no aporta más que uno con 500, y tarda mucho más.
- **Usa `debounce=True`** en los campos de texto.
- **Usa `State` en lugar de `Input`** para lo que no deba disparar el callback.
- **Considera callbacks del lado del cliente** para lógica trivial.
- **Pagina las tablas** en lugar de volcar 50.000 filas.


---

# Módulo 8. Construcción guiada de un cuadro de mando

Vamos a construir un cuadro de mando de ventas por capas. Cada versión añade una pieza y explica por qué.

Los datos: 23.307 pedidos de dos años, con región, categoría, producto, canal, unidades, ingresos y beneficio.


## 8.1 Versión 1: el esqueleto y la carga de datos

```python
import os
from dash import Dash, html
import dash_bootstrap_components as dbc
import pandas as pd

DIR = os.path.dirname(os.path.abspath(__file__))

# Se ejecuta UNA vez, al arrancar el servidor.
# En Dash no hace falta caché para esto: el script no se reejecuta.
DF = pd.read_csv(os.path.join(DIR, "ventas.csv"), parse_dates=["fecha"])
DF["mes"] = DF["fecha"].dt.to_period("M").dt.to_timestamp()

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP],
           title="Cuadro de mando de ventas")

server = app.server        # imprescindible para el despliegue (módulo 9)

app.layout = dbc.Container([
    html.H2("Cuadro de mando de ventas"),
    html.P(f"{len(DF):,} pedidos cargados"),
], fluid=True)

if __name__ == "__main__":
    app.run(debug=True)
```

**Lo importante:** la línea `server = app.server`. Es la instancia de Flask que hay bajo Dash, y es lo que gunicorn necesitará en producción. Ponerla desde el principio evita descubrir su falta el día del despliegue.


## 8.2 Versión 2: los filtros

```python
REGIONES = sorted(DF["region"].unique())
CATEGORIAS = sorted(DF["categoria"].unique())
FECHA_MIN, FECHA_MAX = DF["fecha"].min(), DF["fecha"].max()

barra_filtros = dbc.Card(dbc.CardBody([
    dbc.Row([
        dbc.Col([
            html.Label("Periodo", className="fw-semibold small"),
            dcc.DatePickerRange(
                id="f-fechas",
                min_date_allowed=FECHA_MIN, max_date_allowed=FECHA_MAX,
                start_date=FECHA_MIN, end_date=FECHA_MAX,
                display_format="DD/MM/YYYY",
            ),
        ], md=4),
        dbc.Col([
            html.Label("Región", className="fw-semibold small"),
            dcc.Dropdown(id="f-region", options=REGIONES, value=[],
                         multi=True, placeholder="Todas"),
        ], md=4),
        dbc.Col([
            html.Label("Categoría", className="fw-semibold small"),
            dcc.Dropdown(id="f-categoria", options=CATEGORIAS, value=[],
                         multi=True, placeholder="Todas"),
        ], md=4),
    ], className="g-3"),
]), className="shadow-sm mb-4")
```

**Lo importante:** las opciones de los desplegables se generan **a partir de los datos**, no escritas a mano. Si mañana aparece una región nueva en el CSV, el filtro la recoge solo.

## 8.3 Versión 3: la función de filtrado y su memoización

Antes de escribir ningún callback, resolvemos el problema del filtrado compartido:

```python
from functools import lru_cache

def filtrar(df, desde, hasta, regiones, categorias):
    m = (df["fecha"] >= pd.Timestamp(desde)) & (df["fecha"] <= pd.Timestamp(hasta))
    if regiones:
        m &= df["region"].isin(regiones)
    if categorias:
        m &= df["categoria"].isin(categorias)
    return df[m]


@lru_cache(maxsize=64)
def _filtrar_memo(desde, hasta, regiones, categorias):
    return filtrar(DF, desde, hasta, list(regiones), list(categorias))


def datos(desde, hasta, regiones, categorias):
    return _filtrar_memo(
        desde, hasta, tuple(regiones or []), tuple(categorias or [])
    ).copy()
```

**Lo importante:** cada interacción del usuario disparará seis o siete callbacks, y todos necesitan el mismo subconjunto. Sin memoización, el filtrado se repite siete veces. Con ella, se hace una y las otras seis son instantáneas.

Y como vimos en 7.2, **no** usamos `dcc.Store` para esto.

## 8.4 Versión 4: los KPI

```python
def tarjeta_kpi(titulo, id_valor, id_delta, icono):
    """Componente reutilizable: evita repetir el mismo bloque cuatro veces."""
    return dbc.Card(dbc.CardBody([
        html.Div([
            html.Span(icono, className="fs-4 me-2"),
            html.Span(titulo, className="text-muted small text-uppercase"),
        ]),
        html.H3(id=id_valor, className="mb-0 mt-2 fw-bold"),
        html.Small(id=id_delta, className="text-muted"),
    ]), className="shadow-sm h-100")


fila_kpis = dbc.Row([
    dbc.Col(tarjeta_kpi("Ingresos", "kpi-ingresos", "kpi-ingresos-d", "💰"), md=3),
    dbc.Col(tarjeta_kpi("Beneficio", "kpi-beneficio", "kpi-beneficio-d", "📈"), md=3),
    dbc.Col(tarjeta_kpi("Pedidos", "kpi-pedidos", "kpi-pedidos-d", "🧾"), md=3),
    dbc.Col(tarjeta_kpi("Ticket medio", "kpi-ticket", "kpi-ticket-d", "🛒"), md=3),
], className="g-3 mb-4")
```

Y el callback. Como los cinco filtros se repiten en casi todos los callbacks, los declaramos una vez:

```python
FILTROS = [
    Input("f-fechas", "start_date"),
    Input("f-fechas", "end_date"),
    Input("f-region", "value"),
    Input("f-categoria", "value"),
]


@callback(
    Output("kpi-ingresos", "children"), Output("kpi-ingresos-d", "children"),
    Output("kpi-beneficio", "children"), Output("kpi-beneficio-d", "children"),
    Output("kpi-pedidos", "children"), Output("kpi-pedidos-d", "children"),
    Output("kpi-ticket", "children"), Output("kpi-ticket-d", "children"),
    FILTROS,
)
def actualizar_kpis(desde, hasta, regiones, categorias):
    d = datos(desde, hasta, regiones, categorias)
    if d.empty:
        return ("—", "sin datos") * 4

    ingresos = d["ingresos"].sum()
    beneficio = d["beneficio"].sum()
    pedidos = len(d)
    ticket = ingresos / pedidos if pedidos else 0

    return (
        formato_euros(ingresos), f"{d['unidades'].sum():,} unidades",
        formato_euros(beneficio), f"margen {beneficio/ingresos:.1%}",
        f"{pedidos:,}", f"{pedidos/24:.0f} al mes de media",
        formato_euros(ticket), f"{d['unidades'].mean():.1f} uds. por pedido",
    )
```

**Lo importante:**

- Definir `FILTROS` como lista reutilizable evita erratas y hace visible la dependencia.
- La función devuelve **ocho valores** porque hay ocho `Output`. El orden importa.
- El caso `d.empty` se maneja explícitamente: sin eso, la división por cero rompería la aplicación cuando los filtros no devuelven nada.

Y una función de formato, porque «9810445.32» no es una cifra presentable:

```python
def formato_euros(valor):
    if abs(valor) >= 1_000_000:
        return f"{valor/1_000_000:.2f} M€".replace(".", ",")
    if abs(valor) >= 1_000:
        return f"{valor/1_000:.1f} k€".replace(".", ",")
    return f"{valor:.0f} €"
```

## 8.5 Versión 5: los gráficos

```python
@callback(Output("g-evolucion", "figure"), FILTROS)
def grafico_evolucion(desde, hasta, regiones, categorias):
    d = datos(desde, hasta, regiones, categorias)
    if d.empty:
        return figura_vacia()

    agg = d.groupby("mes", as_index=False).agg(
        ingresos=("ingresos", "sum"), beneficio=("beneficio", "sum")
    )

    fig = go.Figure()
    fig.add_bar(x=agg["mes"], y=agg["ingresos"], name="Ingresos",
                marker_color="#2563eb", opacity=0.85)
    fig.add_scatter(x=agg["mes"], y=agg["beneficio"], name="Beneficio",
                    mode="lines+markers", line=dict(color="#16a34a", width=3))
    fig.update_layout(
        template="plotly_white", margin=dict(l=10, r=10, t=10, b=10),
        legend=dict(orientation="h", y=1.12, x=0), hovermode="x unified",
    )
    return fig
```

**Lo importante:** el `groupby` agrega de 23.000 filas a 24 puntos antes de dibujar. Enviar 23.000 puntos al navegador sería lento y no se vería mejor.

## 8.6 Versión 6: filtrado cruzado

Aquí está la funcionalidad que distingue un cuadro de mando de una colección de gráficos: hacer clic en el gráfico de categorías filtra la tabla.

```python
@callback(
    Output("tabla", "rowData"),
    Output("txt-filtro-tabla", "children"),
    FILTROS + [Input("g-categoria", "clickData")],
)
def actualizar_tabla(desde, hasta, regiones, categorias, clic):
    d = datos(desde, hasta, regiones, categorias)
    if d.empty:
        return [], "Sin datos"

    aviso = ""
    if clic:
        categoria = clic["points"][0]["label"]
        d = d[d["categoria"] == categoria]
        aviso = f"Filtrado por categoría: {categoria}"

    cols = ["id_pedido", "fecha", "region", "categoria", "producto", "ingresos"]
    d = d[cols].copy()
    d["fecha"] = pd.to_datetime(d["fecha"]).dt.strftime("%d/%m/%Y")

    return d.to_dict("records"), aviso
```

**Lo importante:**

- La estructura de `clickData` depende del tipo de gráfico. En un gráfico de tarta, la categoría está en `points[0]["label"]`; en uno de barras, en `points[0]["x"]`. Averígualo imprimiéndola.
- El aviso de texto es esencial: sin él, el usuario ve una tabla filtrada y no entiende por qué.


## 8.7 Versión 7: botón de reinicio y descarga

```python
@callback(
    Output("f-fechas", "start_date"), Output("f-fechas", "end_date"),
    Output("f-region", "value"), Output("f-categoria", "value"),
    Input("btn-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset(_):
    return FECHA_MIN, FECHA_MAX, [], []


@callback(
    Output("descarga", "data"),
    Input("btn-descargar", "n_clicks"),
    State("f-fechas", "start_date"), State("f-fechas", "end_date"),
    State("f-region", "value"), State("f-categoria", "value"),
    prevent_initial_call=True,
)
def descargar(_, desde, hasta, regiones, categorias):
    d = datos(desde, hasta, regiones, categorias)
    if d.empty:
        return dash.no_update
    return dcc.send_data_frame(d.to_csv, "ventas_filtradas.csv", index=False)
```

**Lo importante:** el callback de descarga usa `State`, no `Input`, para los filtros. Con `Input`, se descargaría un fichero cada vez que el usuario tocara un filtro.

## 8.8 Estructura final

```
1. Imports
2. Carga de datos (nivel de módulo)
3. Constantes derivadas (opciones de los filtros, paleta)
4. app = Dash(...) y server = app.server
5. Funciones auxiliares (filtrado, memoización, formato)
6. Componentes reutilizables (tarjeta_kpi, figura_vacia)
7. Layout
8. FILTROS (lista de Inputs reutilizable)
9. Callbacks
10. if __name__ == "__main__": app.run(debug=True)
```

## 8.9 Rendimiento medido

Sobre el ejemplo real, con 23.307 filas:

| Callback | Tiempo |
|---|---|
| Resumen de filtros (primera llamada) | 100 ms |
| KPIs (memoización activa) | **7 ms** |
| Gráfico de evolución | 499 ms |
| Gráfico de categorías | 123 ms |
| Tabla completa | 891 ms |
| Filtrado cruzado | 127 ms |

El salto de 100 ms a 7 ms entre el primer callback y el segundo es la memoización funcionando: el primero hace el filtrado, los demás lo reutilizan.

---

# Módulo 9. Publicación en internet

Aquí está la diferencia práctica más importante frente a Streamlit: **Dash no se puede desplegar en Streamlit Community Cloud**. Es una aplicación Flask, así que necesita un alojamiento que sepa servir aplicaciones WSGI.

## 9.1 Qué cambia respecto a Streamlit

| | Streamlit | Dash |
|---|---|---|
| Servidor de producción | Incluido | **gunicorn** (hay que añadirlo) |
| Línea imprescindible | — | `server = app.server` |
| Alojamiento gratuito | Community Cloud | Render, HF Spaces (Docker), Fly.io |
| Fichero de arranque | `streamlit run app.py` | `gunicorn app:server` |

## 9.2 Preparar la aplicación

### Paso 1: exponer el servidor Flask

En `app.py`, después de crear la aplicación:

```python
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server          # <-- ESTA LÍNEA
```

`app.server` es la instancia de Flask que Dash usa por debajo. Gunicorn no sabe nada de Dash: necesita un objeto WSGI, y ese objeto es `server`.

Cuando ejecutas `gunicorn app:server`, gunicorn lee: «importa el módulo `app` y sirve el objeto llamado `server`».

**Si olvidas esta línea**, el despliegue falla con:

```
Failed to find attribute 'server' in 'app'
```

### Paso 2: proteger el bloque de desarrollo

```python
if __name__ == "__main__":
    app.run(debug=True)
```

Este bloque **solo se ejecuta con `python app.py`**. Gunicorn importa el módulo, así que `__name__` no vale `"__main__"` y el bloque se ignora. Es lo que queremos: en producción manda gunicorn, no `app.run`.

Y nunca dejes `debug=True` accesible en producción: expone una consola de depuración que permite ejecutar código arbitrario.

### Paso 3: `requirements.txt`

```
dash[ag-grid]>=4.4,<5
dash-bootstrap-components>=2.0,<3
plotly>=6.0
pandas>=2.2
gunicorn>=23.0
```

**`gunicorn` debe estar aquí.** Es la omisión más frecuente: la aplicación construye bien y luego falla al arrancar porque el comando `gunicorn` no existe.

Genera el fichero desde un entorno virtual limpio, o escríbelo a mano. `pip freeze` desde el entorno global produce cientos de líneas inútiles.

### Paso 4: `Procfile`

Un fichero de una línea, sin extensión, que indica cómo arrancar:

```
web: gunicorn app:server --workers 2 --timeout 120
```

Desglose del comando:

| Parte | Significado |
|---|---|
| `web:` | Tipo de proceso (lo esperan Render y Heroku) |
| `gunicorn` | El servidor WSGI |
| `app:server` | Módulo `app.py`, objeto `server` |
| `--workers 2` | Dos procesos para atender peticiones en paralelo |
| `--timeout 120` | Segundos antes de matar una petición colgada |

Sobre `--workers`: cada uno es un proceso independiente con su **propia copia de los datos en memoria**. Con un DataFrame de 500 MB y 4 workers, son 2 GB. En alojamiento gratuito, 1 o 2 workers.

Y ojo con la memoria: si usas `lru_cache`, cada worker tiene la suya. Para caché compartida hace falta Redis (sección 7.3).

## 9.3 Opción A: Render (la recomendada)

Render tiene capa gratuita y es lo más parecido a lo que era Heroku.

### Paso 1: subir a GitHub

Tu repositorio debe contener:

```
mi-dashboard/
├── app.py                ← con 'server = app.server'
├── requirements.txt      ← con gunicorn
├── Procfile
├── ventas.csv            ← los datos, si son pequeños
├── .gitignore
└── assets/               ← CSS opcional
```

```bash
git init
git add .
git commit -m "Cuadro de mando inicial"
git branch -M main
git remote add origin https://github.com/TU_USUARIO/mi-dashboard.git
git push -u origin main
```

Recuerda que GitHub ya no acepta la contraseña de la cuenta: necesitas un *Personal Access Token* (Settings → Developer settings → Personal access tokens → Tokens classic, con permiso `repo`).

### Paso 2: crear el servicio en Render

1. Entra en [render.com](https://render.com) y regístrate con GitHub.
2. **New +** → **Web Service**.
3. Conecta tu repositorio.
4. Rellena:

| Campo | Valor |
|---|---|
| **Name** | `mi-dashboard` (será parte de la URL) |
| **Region** | Frankfurt, si estás en Europa |
| **Branch** | `main` |
| **Runtime** | Python 3 |
| **Build Command** | `pip install -r requirements.txt` |
| **Start Command** | `gunicorn app:server --workers 2 --timeout 120` |
| **Instance Type** | Free |

5. En **Advanced**, añade una variable de entorno:

```
PYTHON_VERSION = 3.11
```

Sin esto, Render elige una versión que puede no ser compatible con tus dependencias.

6. **Create Web Service**.

Verás el log de construcción en directo. En cinco o diez minutos tendrás:

```
https://mi-dashboard.onrender.com
```

### Paso 3 (opcional): `render.yaml`

En lugar de rellenar el formulario, puedes describir el servicio en un fichero:

```yaml
services:
  - type: web
    name: cuadro-mando-ventas
    runtime: python
    plan: free
    buildCommand: pip install -r requirements.txt
    startCommand: gunicorn app:server --workers 2 --timeout 120
    envVars:
      - key: PYTHON_VERSION
        value: "3.11"
```

Con este fichero en el repositorio, Render lo detecta y configura todo solo. Y queda versionado, que es la ventaja real.

### Actualizar

Cada `git push` redespliega automáticamente.

### Límites de la capa gratuita

| Aspecto | Detalle |
|---|---|
| Memoria | 512 MB |
| Hibernación | **La aplicación se duerme tras 15 minutos sin visitas** |
| Reactivación | El primer visitante espera 30-60 segundos |
| Horas | Cuota mensual limitada |

La hibernación es el inconveniente serio. Si vas a enseñar el panel en una presentación, ábrelo cinco minutos antes.

Y **512 MB es poco**: vigila el tamaño de los datos que cargas en memoria.

## 9.4 Opción B: Hugging Face Spaces con Docker

Spaces no tiene SDK nativo para Dash, pero admite Docker, y eso sirve para cualquier cosa.

`Dockerfile`:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# Hugging Face Spaces EXIGE el puerto 7860
ENV PORT=7860
EXPOSE 7860

# 0.0.0.0 es imprescindible dentro de un contenedor
CMD gunicorn app:server --bind 0.0.0.0:$PORT --workers 2 --timeout 120
```

`README.md` con la cabecera YAML:

```yaml
---
title: Cuadro de mando de ventas
emoji: 📊
colorFrom: blue
colorTo: green
sdk: docker
app_port: 7860
pinned: false
---
```

Dos detalles que hacen fallar esto habitualmente:

- **El puerto debe ser 7860** y declararse en `app_port`.
- **`--bind 0.0.0.0`**, no `127.0.0.1`. Dentro de un contenedor, `127.0.0.1` solo escucha a sí mismo y la aplicación resulta inaccesible desde fuera.

Ventaja frente a Render: más memoria en la capa CPU Basic. Inconveniente: conviene verificar las condiciones vigentes del plan gratuito, que han cambiado.

## 9.5 Opción C: Docker en cualquier sitio

El mismo `Dockerfile` sirve para un VPS, Google Cloud Run, Fly.io o Railway:

```bash
docker build -t mi-dashboard .
docker run -p 8050:7860 mi-dashboard
```

Para Cloud Run, el puerto lo inyecta la plataforma en `$PORT`, y el `CMD` de arriba ya lo respeta.


## 9.6 Opción D: Plotly Cloud

Plotly ofrece su propio servicio de alojamiento para aplicaciones Dash, con publicación en un clic desde Plotly Studio y funciones de colaboración por equipos. Los planes premium permiten dominio propio. Es la vía más integrada si trabajas dentro del ecosistema de Plotly; conviene consultar las condiciones actuales en su web.

## 9.7 Comparativa

| Plataforma | Gratis | Memoria | Dificultad | Cuándo |
|---|---|---|---|---|
| **Render** | Sí (hiberna) | 512 MB | Baja | Opción por defecto |
| **HF Spaces (Docker)** | Verificar plan | Mayor | Media | Si ya estás en HF |
| **Fly.io** | Crédito | Configurable | Media | Control fino |
| **Railway** | Crédito | Configurable | Baja | Con base de datos |
| **Google Cloud Run** | Capa gratuita | Configurable | Media-alta | Producción real |
| **Plotly Cloud** | Verificar | — | Muy baja | Ecosistema Plotly |
| **VPS propio** | No | La que pagues | Alta | Datos sensibles |

## 9.8 Errores frecuentes en el despliegue

| Mensaje | Causa | Solución |
|---|---|---|
| `Failed to find attribute 'server' in 'app'` | Falta `server = app.server` | Añádela tras crear la app |
| `gunicorn: command not found` | No está en `requirements.txt` | Añádelo |
| `ModuleNotFoundError` | Dependencia ausente | Añádela a `requirements.txt` |
| `Could not find a version that satisfies...` | Versión incompatible con el Python del servidor | Fija `PYTHON_VERSION=3.11` |
| Arranca y muere | Escucha en `127.0.0.1` | Usa `--bind 0.0.0.0:$PORT` |
| `Worker timeout` | Callback demasiado lento | Sube `--timeout`, o usa background callbacks |
| `MemoryError` / se reinicia solo | Datos demasiado grandes | Reduce workers, agrega los datos |
| `FileNotFoundError` con el CSV | Ruta relativa al directorio de trabajo | Usa `os.path.dirname(os.path.abspath(__file__))` |
| Funciona en local, falla desplegado | Dependencia instalada solo en tu máquina | Prueba en un entorno virtual limpio |

**La comprobación previa que evita casi todo:**

```bash
python -m venv test_env
source test_env/bin/activate
pip install -r requirements.txt
gunicorn app:server --bind 127.0.0.1:8050
```

Fíjate en que se prueba con **gunicorn**, no con `python app.py`. Es lo que hará el servidor, y hay fallos (como la falta de `server = app.server`) que solo aparecen así.

---

# Módulo 10. Buenas prácticas y catálogo de errores

## 10.1 Organización del código

Para aplicaciones que crecen, separa en módulos:

```
mi-dashboard/
├── app.py              # Dash(), server, layout principal
├── datos.py            # carga y filtrado
├── graficos.py         # funciones que devuelven figuras
├── componentes.py      # tarjetas, cabeceras reutilizables
├── callbacks.py        # los callbacks
└── assets/
    └── estilos.css
```

Las funciones que construyen figuras no deberían saber nada de Dash: reciben un DataFrame y devuelven una figura. Así se pueden probar sin arrancar el servidor:

```python
# graficos.py
def evolucion_mensual(df):
    agg = df.groupby("mes", as_index=False)["ingresos"].sum()
    return px.line(agg, x="mes", y="ingresos")

# test_graficos.py
def test_evolucion():
    fig = evolucion_mensual(df_prueba)
    assert len(fig.data) == 1
```

## 10.2 Diseño del cuadro de mando

- **Pirámide invertida:** KPIs arriba, tendencias en medio, detalle abajo.
- **Filtros en un sitio fijo**, arriba o en un lateral, nunca dispersos.
- **Colores coherentes**: la misma categoría, el mismo color en todos los gráficos.
- **Formatea las cifras.** «9,81 M€» se lee; «9810445.32» no.
- **Estado vacío explícito.** Si los filtros no devuelven nada, dilo con un mensaje, no con un gráfico roto.
- **Indica qué filtros están activos.** Un usuario que ve cifras extrañas debe poder saber por qué.

## 10.3 Los diez errores más frecuentes

**1. Olvidar `server = app.server`.** Funciona en local, falla en el despliegue.

**2. Descuadre entre `Output` y valores devueltos.** Ocho `Output` exigen ocho valores.

**3. Meter DataFrames grandes en `dcc.Store`.** Sección 7.2.

**4. Usar `dash_table.DataTable`.** Deprecado en Dash 4; usa AG Grid.

**5. Usar `app.run_server()`.** Eliminado en Dash 3; es `app.run()`.

**6. `Input` donde debería ir `State`.** Provoca callbacks que se disparan sin motivo.

**7. Olvidar `debounce=True`** en campos de texto: un callback por tecla.

**8. `id` duplicados.** Dash falla al arrancar con un mensaje poco claro.

**9. No manejar el DataFrame vacío.** División por cero cuando los filtros no devuelven nada.

**10. Dejar `debug=True` en producción.** Riesgo de seguridad serio.

---

# Apéndice A. Chuleta de referencia

## Esqueleto

```python
import os
from functools import lru_cache

import dash
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, State, callback, dcc, html

DIR = os.path.dirname(os.path.abspath(__file__))
DF = pd.read_csv(os.path.join(DIR, "datos.csv"))

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
server = app.server                      # para el despliegue

app.layout = dbc.Container([
    dcc.Dropdown(id="filtro", options=sorted(DF["cat"].unique()), value=None),
    dcc.Graph(id="grafico"),
], fluid=True)


@callback(Output("grafico", "figure"), Input("filtro", "value"))
def actualizar(cat):
    d = DF if not cat else DF[DF["cat"] == cat]
    return px.bar(d, x="mes", y="total")


if __name__ == "__main__":
    app.run(debug=True)
```

## Componentes

```python
html.Div, html.H1..H6, html.P, html.Span, html.A, html.Img, html.Br, html.Hr

dcc.Dropdown, dcc.Slider, dcc.RangeSlider, dcc.Input, dcc.Textarea
dcc.RadioItems, dcc.Checklist, dcc.DatePickerSingle, dcc.DatePickerRange
dcc.Upload, dcc.Graph, dcc.Store, dcc.Download, dcc.Interval
dcc.Location, dcc.Loading, dcc.Tabs, dcc.Markdown

dbc.Container, dbc.Row, dbc.Col, dbc.Card, dbc.CardBody
dbc.Button, dbc.Alert, dbc.Tabs, dbc.NavbarSimple, dbc.Spinner

dag.AgGrid          # tablas (DataTable está deprecado)
```

## Callbacks

```python
@callback(
    Output("id", "propiedad"),
    Input("id", "propiedad"),
    State("id", "propiedad"),
    prevent_initial_call=True,
)
def funcion(entrada, estado):
    return valor

raise PreventUpdate          # cancelar todo el callback
return dash.no_update        # no actualizar una salida concreta
ctx.triggered_id             # qué componente disparó el callback
allow_duplicate=True         # dos callbacks a la misma salida
background=True              # tarea larga en segundo plano
```

## Propiedades habituales

```python
Dropdown/Slider/Input  -> value
DatePickerRange        -> start_date, end_date
Button                 -> n_clicks
Graph                  -> figure (salida) · clickData, selectedData (entrada)
Upload                 -> contents, filename
AgGrid                 -> rowData (salida) · selectedRows (entrada)
html.*                 -> children
```

## Despliegue

```python
server = app.server
```

```
# requirements.txt
dash[ag-grid]>=4.4,<5
gunicorn>=23.0

# Procfile
web: gunicorn app:server --workers 2 --timeout 120
```

```bash
gunicorn app:server --bind 0.0.0.0:$PORT --workers 2 --timeout 120
```


---

# Apéndice B. Comparativa Dash / Streamlit / Gradio

| | Dash | Streamlit | Gradio |
|---|---|---|---|
| Modelo mental | Layout + callbacks | Script que se reejecuta | Función con interfaz |
| Ejecución del script | Una vez, al arrancar | En cada interacción | Una vez |
| Reactividad | Explícita (callbacks) | Automática | Explícita (eventos) |
| Caché | Rara vez necesaria | Imprescindible | Rara vez necesaria |
| Estado | `dcc.Store` | `st.session_state` | `gr.State` |
| Tablas | AG Grid | `st.dataframe` | `gr.Dataframe` |
| Gráficos | Plotly | Cualquiera | Cualquiera |
| Arranque local | `python app.py` | `streamlit run app.py` | `python app.py` |
| Producción | `gunicorn app:server` | Incluido | Incluido |
| Alojamiento gratuito | Render, Docker | Community Cloud | HF Spaces |
| Verbosidad | Alta | Muy baja | Media |
| Control del diseño | Total | Limitado | Medio |
| Multipágina | Nativo (`pages/`) | Nativo (`pages/`) | Manual |
| Curva de aprendizaje | Moderada | Muy suave | Suave |

## Cuándo elegir cada uno

**Dash** cuando el resultado sea un cuadro de mando que alguien vaya a usar a diario, cuando necesites control del diseño, cuando haya muchos controles interactuando entre sí, o cuando deba integrarse en infraestructura corporativa.

**Streamlit** para prototipos, informes interactivos y aplicaciones de datos que debas tener listas hoy.

**Gradio** para demostrar un modelo: entra un dato, sale una predicción. Sus componentes multimedia no tienen rival.

---

## Recursos

- Documentación: `dash.plotly.com`
- Referencia de componentes: `dash.plotly.com/dash-core-components`
- Bootstrap components: `dash-bootstrap-components.opensource.faculty.ai`
- AG Grid: `dash.plotly.com/dash-ag-grid`
- Plotly Express: `plotly.com/python/plotly-express`
- Foro: `community.plotly.com`
- Render: `render.com/docs`